In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [2]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
#teams.sort()
#print(teams)

In [3]:
replace_dict = {"SFO":"SF", "NOS":"NO", "GBP":"GB", "KCC": "KC", "NOS": "NO", "SFO":"SF"}
clean_teams = [replace_dict.get(item,item) for item in teams]
#print(clean_teams)

In [4]:
df_odds = dict(zip(clean_teams, odds))
#print(df_odds)
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

{'HOU': '14.75', 'NYG': '17.50', 'NYJ': '19.25', 'JAC': '19.50', 'DET': '19.50', 'PIT': '19.50', 'IND': '20.00', 'NEP': '21.50', 'DEN': '21.75', 'MIA': '22.00', 'CHI': '22.00', 'BAL': '22.25', 'ATL': '23.00', 'PHI': '23.75', 'CAR': '23.75', 'LVR': '24.00', 'NO': '24.50', 'WAS': '24.50', 'SEA': '24.75', 'TEN': '25.25', 'CLE': '25.50', 'ARI': '25.50', 'GB': '25.50', 'MIN': '26.00', 'CIN': '26.50', 'SF': '27.25', 'LAC': '27.50', 'DAL': '27.75', 'TBB': '28.00', 'LAR': '29.00', 'KC': '30.25', 'BUF': '32.25'}


In [8]:
df_fd = pd.read_csv("data/nfl/FanDuel-NFL-2021 ET-10 ET-03 ET-64596-players-list.csv")
# dk_teams = df_dk.TeamAbbrev.unique()
# dk_teams.sort()
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,64596-39280,RB,Derrick,Derrick Henry,Henry,23.933334,3.0,10200,TEN@NYJ,TEN,NYJ,NaN,NaN,NaN,NaN,NaN,RB/FLEX
1,64596-54140,RB,Dalvin,Dalvin Cook,Cook,16.600000,2.0,9500,CLE@MIN,MIN,CLE,Q,Ankle,NaN,NaN,NaN,RB/FLEX
2,64596-42104,RB,Alvin,Alvin Kamara,Kamara,13.633334,3.0,9000,NYG@NO,NO,NYG,NaN,NaN,NaN,NaN,NaN,RB/FLEX
3,64596-57439,QB,Patrick,Patrick Mahomes,Mahomes,27.733332,3.0,8700,KC@PHI,KC,PHI,NaN,NaN,NaN,NaN,NaN,QB
4,64596-79970,WR,Cooper,Cooper Kupp,Kupp,26.233332,3.0,8600,ARI@LAR,LAR,ARI,NaN,NaN,NaN,NaN,NaN,WR/FLEX


In [10]:
df_fd["expts"] = df_fd["Team"].map(df_odds)
df_fd.expts = df_fd.expts.astype('float')
pts_bins = [1.2, 1.1, 1.0, .9,.8]
df_fd["rkpts"] = pd.cut(df_fd.expts, 5, labels=pts_bins)
df_fd.rkpts = df_fd.rkpts.astype('float')
df_fd["new_expts"] = df_fd.FPPG * df_fd.rkpts
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position,expts,rkpts,new_expts
0,64596-39280,RB,Derrick,Derrick Henry,Henry,23.933334,3.0,10200,TEN@NYJ,TEN,NYJ,NaN,NaN,NaN,NaN,NaN,RB/FLEX,25.25,1.0,23.933334
1,64596-54140,RB,Dalvin,Dalvin Cook,Cook,16.600000,2.0,9500,CLE@MIN,MIN,CLE,Q,Ankle,NaN,NaN,NaN,RB/FLEX,26.00,0.9,14.940000
2,64596-42104,RB,Alvin,Alvin Kamara,Kamara,13.633334,3.0,9000,NYG@NO,NO,NYG,NaN,NaN,NaN,NaN,NaN,RB/FLEX,24.50,1.0,13.633334
3,64596-57439,QB,Patrick,Patrick Mahomes,Mahomes,27.733332,3.0,8700,KC@PHI,KC,PHI,NaN,NaN,NaN,NaN,NaN,QB,30.25,0.8,22.186666
4,64596-79970,WR,Cooper,Cooper Kupp,Kupp,26.233332,3.0,8600,ARI@LAR,LAR,ARI,NaN,NaN,NaN,NaN,NaN,WR/FLEX,29.00,0.8,20.986666


In [11]:
df_fd.FPPG = df_fd.new_expts
df_fd = df_fd.drop(['expts', 'rkpts', 'new_expts'], axis=1)
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,64596-39280,RB,Derrick,Derrick Henry,Henry,23.933334,3.0,10200,TEN@NYJ,TEN,NYJ,NaN,NaN,NaN,NaN,NaN,RB/FLEX
1,64596-54140,RB,Dalvin,Dalvin Cook,Cook,14.940000,2.0,9500,CLE@MIN,MIN,CLE,Q,Ankle,NaN,NaN,NaN,RB/FLEX
2,64596-42104,RB,Alvin,Alvin Kamara,Kamara,13.633334,3.0,9000,NYG@NO,NO,NYG,NaN,NaN,NaN,NaN,NaN,RB/FLEX
3,64596-57439,QB,Patrick,Patrick Mahomes,Mahomes,22.186666,3.0,8700,KC@PHI,KC,PHI,NaN,NaN,NaN,NaN,NaN,QB
4,64596-79970,WR,Cooper,Cooper Kupp,Kupp,20.986666,3.0,8600,ARI@LAR,LAR,ARI,NaN,NaN,NaN,NaN,NaN,WR/FLEX


In [14]:
df_fd.to_csv('data/nfl/FDprojections.csv')

In [20]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter

optimizer = get_optimizer(Site.FANDUEL, Sport.FOOTBALL)
optimizer.load_players_from_csv("data/nfl/FDprojections.csv")
for lineup in optimizer.optimize(9, max_exposure=.3):
    print(lineup)
optimizer.export('data/nfl/fd_result.csv')

 1. QB      Daniel Jones                  QB    NYG            NYG@NO   26.952         7000.0$   
 2. RB      Cordarrelle Patterson         RB/WR ATL            WAS@ATL  14.1           6000.0$   
 3. RB      Derrick Henry                 RB    TEN            TEN@NYJ  23.933         10200.0$  
 4. WR      Brandin Cooks                 WR    HOU            HOU@BUF  20.08          6900.0$   
 5. WR      Brandon Zylstra               WR    CAR            CAR@DAL  11.9           4900.0$   
 6. WR      Cooper Kupp                   WR    LAR            ARI@LAR  20.987         8600.0$   
 7. TE      T.J. Hockenson                TE    DET            DET@CHI  14.777         6600.0$   
 8. FLEX    Jamaal Williams               RB    DET            DET@CHI  14.63          5600.0$   
 9. DEF     Denver Broncos                D     DEN            BAL@DEN  13.2           4200.0$   

Fantasy Points 160.56
Salary 60000.00

 1. QB      Daniel Jones                  QB    NYG            NYG@NO   26.952